In [30]:
import fastplotlib as fpl
import os
import masknmf
import sys
import numpy as np

from ipywidgets import HBox, VBox
import math
import torch

##In the existing folder
import iblwfci_vis
import iblwfci_utils

from tqdm import tqdm
from one.api import ONE
from brainbox.io.one import SessionLoader
one = ONE()

%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
"""
The two datasets from FD that we will process are: 
(1) 71ceb3d4-ca68-4380-8fe7-9f63d26222f6 (This is the existing FD_24/2023-06-07/001 data that I had already downloaded)
(2) 8df7b200-e44c-4c67-82e9-2666ba05d649 (This is the  FD_24/2023-06-08/001 data)
"""

# Step 1: Let's load the trial data we need from ONE to do the trial-triggered analysis 

In [20]:
eid = "3158300f-e72c-42fc-ac6c-c981615fe00f"
sl = SessionLoader(eid = eid, one=one)
sl.load_trials()
sl.trials.keys()
stim_ontimes = sl.trials['stimOn_times']

In [19]:
imaging_times = one.load_dataset(eid, "imaging.times.npy", collection="alf/widefield")

In [24]:
trial_indices = np.searchsorted(imaging_times, stim_ontimes)

# Step 2: Let's load the stacks we need (from Joao's method and Amol's method) to study trial-trigged signal

In [27]:
## Let's load the 007 folder data (masknmf processing results)
hemocorr_pmd = masknmf.PMDArray.from_hdf5("felicia_jan_26_007/hemocorr.hdf5")
hemocorr_pmd.to('cuda')
hemocorr_pmd.rescale = True

In [28]:
parent_path = "/data/lab/IBL_Alyx/churchlandlab_ucla/Subjects/FD_24/2023-06-07/001/raw_widefield_data"
u_path = os.path.join(parent_path, "U.npy")
svt_path = os.path.join(parent_path, "SVT.npy")
svtcorr_path = os.path.join(parent_path, "SVTcorr.npy")
frames_avg_path = os.path.join(parent_path, "frames_average.npy")

In [29]:
joao_gcamp, joao_blood, joao_hemocorr = iblwfci_utils.load_joao_results(u_path,
                  svt_path,
                  svtcorr_path,
                  frames_avg_path,
                  functional_channel = 0)

In [35]:
joao_gcamp.rescale = True
joao_blood.rescale = True
joao_hemocorr.rescale = True

joao_gcamp.to('cuda')
joao_blood.to('cuda')
joao_hemocorr.to('cuda')

In [44]:
def get_trial_triggered_stack(my_pmd_movie, trial_indices, before_frames = 20, after_frames = 100, device='cpu'):
    my_pmd_movie.to(device)
    average_movie = torch.zeros((before_frames + after_frames, my_pmd_movie.shape[1], my_pmd_movie.shape[2])).to(device)
    num_trials = trial_indices.shape[0]
    for k in tqdm(range(trial_indices.shape[0])):
        curr_ind = trial_indices[k]
        curr_data = my_pmd_movie.getitem_tensor(slice(curr_ind - before_frames, curr_ind + after_frames))
        curr_data /= num_trials
        average_movie += curr_data 
    return average_movie.cpu().numpy()


In [45]:
joao_trial_avg = get_trial_triggered_stack(joao_hemocorr, trial_indices, device='cuda')

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 856/856 [01:35<00:00,  9.01it/s]


In [46]:
amol_trial_avg = get_trial_triggered_stack(hemocorr_pmd, trial_indices, device='cuda')

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 856/856 [00:17<00:00, 50.35it/s]


In [48]:
joao_trial_avg = joao_trial_avg - np.mean(joao_trial_avg, axis = 0, keepdims = True)
amol_trial_avg = amol_trial_avg - np.mean(amol_trial_avg, axis = 0, keepdims = True)


In [51]:
# View results
iblwfci_vis.raster_view(amol_trial_avg, joao_trial_avg, 5)